# UC1 Performance Benchmarks

```
Summary : Benchmarks the Use Case 1 silver -> gold transformations on different data sizes and with two optimisations: broadcast hint on the category dim and caching of the orders_revenue fact before the five gold aggregations.
Data    : silver.orders_revenue_silver, silver.order_items_revenue_silver, silver.product_catalog_silver
```

## What this notebook produces
A small results table with wall-clock runtime for the full UC1 silver->gold build across dataset sizes and optimisation variants.

In [0]:
import time
from pyspark.sql import functions as F

spark.sql("USE CATALOG data_sentinals")

SILVER_ORDERS_REVENUE = "data_sentinals.silver.orders_revenue_silver"
SILVER_ORDER_ITEMS    = "data_sentinals.silver.order_items_revenue_silver"
SILVER_PRODUCT_CAT    = "data_sentinals.silver.product_catalog_silver"

# Reuse the pure logic functions (same module the DLT files import)
import os, sys
repo_root = os.getcwd().split("Olist_DE_Practise_Jobs")[0] + "Olist_DE_Practise_Jobs"
sys.path.append(os.path.join(repo_root, "utilities"))

from revenue_logic import (
    build_gold_revenue_monthly,
    build_gold_revenue_by_category_month,
    build_gold_revenue_by_state_month,
    build_gold_revenue_by_payment_type_month,
)

In [0]:
def timed(label, action):
    """Time a closure that returns a DataFrame; materialise with .count() so Spark actually executes."""
    start = time.perf_counter()
    df = action()
    rows = df.count()
    elapsed = time.perf_counter() - start
    print(f"  {label:<45s} rows={rows:>10,d}   elapsed={elapsed:6.2f}s")
    return elapsed, rows


def run_full_gold_build(orders_revenue, order_items_revenue, tag):
    """Build all four analytical gold DataFrames and return the total time."""
    print(f"--- variant: {tag} ---")
    t_monthly,   _ = timed("gold_revenue_monthly",                  lambda: build_gold_revenue_monthly(orders_revenue))
    t_category,  _ = timed("gold_revenue_by_category_month",        lambda: build_gold_revenue_by_category_month(orders_revenue, order_items_revenue))
    t_state,     _ = timed("gold_revenue_by_state_month",           lambda: build_gold_revenue_by_state_month(orders_revenue))
    t_payment,   _ = timed("gold_revenue_by_payment_type_month",    lambda: build_gold_revenue_by_payment_type_month(orders_revenue))
    total = t_monthly + t_category + t_state + t_payment
    print(f"  TOTAL for {tag}: {total:.2f}s\n")
    return total

## Baseline - full dataset, no hints

In [0]:
spark.conf.set("spark.databricks.remoteFiltering.blockSelfJoins", "false")

orders_revenue_full       = spark.read.table(SILVER_ORDERS_REVENUE)
order_items_revenue_full  = spark.read.table(SILVER_ORDER_ITEMS)

full_rows = orders_revenue_full.count()
items_rows = order_items_revenue_full.count()
print(f"orders_revenue rows      = {full_rows:,}")
print(f"order_items_revenue rows = {items_rows:,}")

baseline_total = run_full_gold_build(orders_revenue_full, order_items_revenue_full, "FULL / baseline")

## Downsampled runs (0.5 and 0.1)

Demonstrate how the gold build scales with input size. `seed=42` keeps results reproducible.

In [0]:
orders_revenue_50   = orders_revenue_full.sample(fraction=0.5, seed=42)
order_items_rev_50  = order_items_revenue_full.sample(fraction=0.5, seed=42)

orders_revenue_10   = orders_revenue_full.sample(fraction=0.1, seed=42)
order_items_rev_10  = order_items_revenue_full.sample(fraction=0.1, seed=42)

sample50_total = run_full_gold_build(orders_revenue_50, order_items_rev_50, "0.5 sample")
sample10_total = run_full_gold_build(orders_revenue_10, order_items_rev_10, "0.1 sample")

## Optimisation 1 - Broadcast the product-catalog dim

`product_catalog_silver` is a small dimension (~32k rows). Forcing a broadcast join avoids a shuffle when enriching `raw_order_items`. The category gold build reads the already-built `order_items_revenue_silver`, so the effect is visible when we rebuild `order_items_revenue` from raw inputs. Below we simulate that join path to isolate the impact.

In [0]:
raw_items   = spark.read.table("data_sentinals.bronze.raw_order_items")
catalog_dim = spark.read.table(SILVER_PRODUCT_CAT).select("product_id", "product_category_en")

# Without broadcast hint
t_no_bc, _ = timed("items <-> catalog join (NO broadcast)",
                    lambda: raw_items.join(catalog_dim, on="product_id", how="left"))

# With broadcast hint
t_bc, _ = timed("items <-> catalog join (broadcast dim)",
                  lambda: raw_items.join(F.broadcast(catalog_dim), on="product_id", how="left"))

## Optimisation 2 - Cache orders_revenue fact

The five gold marts all read `orders_revenue_silver`. Caching it once avoids repeated table reads.

In [0]:
orders_revenue_cached = spark.read.table(SILVER_ORDERS_REVENUE).cache()
orders_revenue_cached.count()
order_items_revenue_cached = spark.read.table(SILVER_ORDER_ITEMS).cache()
order_items_revenue_cached.count()

cached_total = run_full_gold_build(orders_revenue_cached, order_items_revenue_cached, "FULL / cached fact")

orders_revenue_cached.unpersist()
order_items_revenue_cached.unpersist()

## Results summary

In [0]:
import pandas as pd

results = pd.DataFrame(
    [
        ("gold build - full, baseline",       full_rows,                 baseline_total),
        ("gold build - 0.5 sample",           int(full_rows * 0.5),      sample50_total),
        ("gold build - 0.1 sample",           int(full_rows * 0.1),      sample10_total),
        ("gold build - full, cached fact",    full_rows,                 cached_total),
        ("items<->catalog join no broadcast", None,                      t_no_bc),
        ("items<->catalog join broadcasted",  None,                      t_bc),
    ],
    columns=["variant", "rows_in", "elapsed_s"],
)
results["elapsed_s"] = results["elapsed_s"].round(2)
display(results)